# 📈 Project 06 – Stock Trend Predictor

**What you'll learn:** Time series analysis, moving averages, crossover trading strategies, and a machine learning classifier for short-term trend prediction.

| Detail | Info |
|--------|------|
| **Difficulty** | ⭐⭐ Intermediate |
| **Estimated time** | 60–90 minutes |
| **Key libraries** | `numpy`, `pandas`, `matplotlib`, `scikit-learn` |
| **Dataset** | Synthetic stock data (no API keys needed) |

---

## 🎯 Learning Goals
1. Understand what **Simple Moving Average (SMA)** and **Exponential Moving Average (EMA)** are
2. Build a **crossover signal strategy** (buy/sell triggers)
3. Engineer **time-series features** for machine learning
4. Train a **Gradient Boosting classifier** to predict 5-day price direction
5. Visualise results with multi-panel matplotlib charts

## Step 1 — Install & Import Libraries

We only need standard data science libraries. Run this cell once to install them.

In [ ]:
# Uncomment the line below if you haven't installed these yet
# !pip install numpy pandas matplotlib scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

print("✅ Libraries loaded successfully!")

## Step 2 — Generate Synthetic Stock Data

Real stock data requires an API key. Instead, we simulate a realistic price series using **Geometric Brownian Motion (GBM)** — the same mathematical model behind the famous Black-Scholes options formula.

**Key idea:** Each day's price = yesterday's price × e^(drift + random shock)

In [ ]:
def generate_stock_data(start='2022-01-01', periods=500, seed=42):
    """Generate synthetic daily OHLCV stock data using Geometric Brownian Motion."""
    np.random.seed(seed)
    dates = pd.bdate_range(start=start, periods=periods)  # business days only

    mu    = 0.0003   # average daily drift (slight upward trend)
    sigma = 0.015    # daily volatility (1.5%)

    log_returns = np.random.normal(mu, sigma, periods)
    prices = 100.0 * np.exp(np.cumsum(log_returns))  # start at $100

    noise = np.random.uniform(0.002, 0.008, periods)
    df = pd.DataFrame({
        'close':  prices,
        'open':   prices * (1 + np.random.uniform(-0.003, 0.003, periods)),
        'high':   prices * (1 + noise),
        'low':    prices * (1 - noise),
        'volume': np.random.randint(500_000, 5_000_000, periods),
    }, index=dates)
    df.index.name = 'date'
    return df

df = generate_stock_data()
print(f"Generated {len(df)} trading days of data")
print(f"Date range : {df.index[0].date()} → {df.index[-1].date()}")
print(f"Price range: ${df['close'].min():.2f} – ${df['close'].max():.2f}")
df.head()

## Step 3 — Compute Moving Averages

A **moving average** smooths out price noise by averaging the last N closing prices.

| Type | Formula | Characteristic |
|------|---------|----------------|
| **SMA** | mean of last N closes | Equal weight to all past prices |
| **EMA** | weighted mean, recent prices count more | Reacts faster to recent changes |

We use two windows:
- **Short (20 days):** captures recent momentum
- **Long (50 days):** represents the broader trend

In [ ]:
SHORT, LONG = 20, 50

# Simple Moving Averages
df[f'sma_{SHORT}'] = df['close'].rolling(window=SHORT).mean()
df[f'sma_{LONG}']  = df['close'].rolling(window=LONG).mean()

# Exponential Moving Averages
df[f'ema_{SHORT}'] = df['close'].ewm(span=SHORT, adjust=False).mean()
df[f'ema_{LONG}']  = df['close'].ewm(span=LONG,  adjust=False).mean()

print("Moving averages computed. First few rows with NaN (not enough data yet):")
df[['close', 'sma_20', 'sma_50', 'ema_20', 'ema_50']].head(55).tail(10)

## Step 4 — Build Crossover Trading Signals

The **SMA Crossover Strategy** is one of the oldest algorithmic trading techniques:

- 🟢 **BUY** when the short SMA crosses **above** the long SMA → short-term momentum turning bullish
- 🔴 **SELL** when the short SMA crosses **below** the long SMA → short-term momentum turning bearish

In [ ]:
prev_short = df[f'sma_{SHORT}'].shift(1)
prev_long  = df[f'sma_{LONG}'].shift(1)

df['signal'] = 0
df.loc[(df[f'sma_{SHORT}'] > df[f'sma_{LONG}']) & (prev_short <= prev_long), 'signal'] = 1   # Buy
df.loc[(df[f'sma_{SHORT}'] < df[f'sma_{LONG}']) & (prev_short >= prev_long), 'signal'] = -1  # Sell

buy_count  = (df['signal'] ==  1).sum()
sell_count = (df['signal'] == -1).sum()
print(f"🟢 Buy  signals: {buy_count}")
print(f"🔴 Sell signals: {sell_count}")

## Step 5 — Engineer ML Features

We'll turn the raw price data into **numeric features** that a machine learning model can learn from.

In [ ]:
daily_ret = df['close'].pct_change()

# How far is price from each moving average? (normalised)
df['dist_sma_short'] = (df['close'] - df[f'sma_{SHORT}']) / df[f'sma_{SHORT}']
df['dist_sma_long']  = (df['close'] - df[f'sma_{LONG}'])  / df[f'sma_{LONG}']

# Momentum: how much did price change over the last N days?
df['mom_5']  = df['close'].pct_change(5)
df['mom_10'] = df['close'].pct_change(10)

# Rolling volatility (standard deviation of daily returns)
df['volatility_10'] = daily_ret.rolling(10).std()

# SMA crossover ratio
df['sma_cross_ratio'] = (df[f'sma_{SHORT}'] - df[f'sma_{LONG}']) / df[f'sma_{LONG}']

# Target label: 1 if price is higher 5 days from now, else 0
df['target'] = (df['close'].shift(-5) > df['close']).astype(int)

FEATURE_COLS = ['dist_sma_short', 'dist_sma_long', 'mom_5', 'mom_10',
                'volatility_10', 'sma_cross_ratio']

clean_df = df[FEATURE_COLS + ['target']].dropna()
print(f"Clean rows for ML: {len(clean_df)}")
print(f"Target balance — Up: {clean_df['target'].mean():.1%}, Down: {1-clean_df['target'].mean():.1%}")
clean_df[FEATURE_COLS].describe().round(4)

## Step 6 — Train the Classifier

We use **Gradient Boosting** — an ensemble method that builds many small decision trees, each correcting the errors of the previous one.

**Important:** We do NOT shuffle the data before splitting because time-series data has temporal dependencies.

In [ ]:
X = clean_df[FEATURE_COLS].values
y = clean_df['target'].values

# Split WITHOUT shuffling — respect the time order
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")

# Scale features to zero mean and unit variance
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Train model
model = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\n" + classification_report(y_test, y_pred, target_names=['Down (0)', 'Up (1)']))

## Step 7 — Feature Importance

Which features does the model rely on most?

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importances – Gradient Boosting', fontweight='bold')
ax.set_xlabel('Importance Score')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

for feat, imp in importances.sort_values(ascending=False).items():
    print(f"  {feat:<22} {imp:.4f}")

## Step 8 — Visualise the Full Strategy

In [ ]:
plot_df = df.dropna(subset=['sma_20', 'sma_50']).tail(200)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
fig.suptitle('Stock Trend Predictor – SMA Crossover Strategy', fontsize=14, fontweight='bold')

ax1.plot(plot_df.index, plot_df['close'],  color='steelblue', lw=1.5, label='Close Price')
ax1.plot(plot_df.index, plot_df['sma_20'], color='orange',    lw=1.2, linestyle='--', label='SMA 20')
ax1.plot(plot_df.index, plot_df['sma_50'], color='purple',    lw=1.2, linestyle='--', label='SMA 50')
ax1.plot(plot_df.index, plot_df['ema_20'], color='green',     lw=0.8, linestyle=':',  label='EMA 20', alpha=0.7)

buys  = plot_df[plot_df['signal'] ==  1]
sells = plot_df[plot_df['signal'] == -1]
ax1.scatter(buys.index,  buys['close'],  marker='^', color='limegreen', s=120, zorder=5, label='Buy')
ax1.scatter(sells.index, sells['close'], marker='v', color='red',       s=120, zorder=5, label='Sell')
ax1.set_ylabel('Price ($)')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(alpha=0.3)

colors = ['limegreen' if r >= o else 'red'
          for r, o in zip(plot_df['close'], plot_df['open'])]
ax2.bar(plot_df.index, plot_df['volume'] / 1_000_000, color=colors, alpha=0.7)
ax2.set_ylabel('Volume (M)')
ax2.grid(alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('stock_trend_predictor.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved as stock_trend_predictor.png")

## Step 9 — Predict the Next 5 Days

In [ ]:
latest = df[FEATURE_COLS].dropna().tail(1)
latest_scaled = scaler.transform(latest)
pred  = model.predict(latest_scaled)[0]
proba = model.predict_proba(latest_scaled)[0]

trend = '📈 UP' if pred == 1 else '📉 DOWN'
print(f"Last close price : ${df['close'].iloc[-1]:.2f}")
print(f"5-day prediction : {trend}")
print(f"Confidence       : {max(proba):.1%}")
print(f"(Down prob={proba[0]:.1%}, Up prob={proba[1]:.1%})")

---

## 🧪 Try It Yourself!

Here are some experiments to deepen your understanding:

1. **Change the moving average windows** — try `SHORT=10, LONG=30`. Does accuracy improve?
2. **Add more features** — try including the RSI (Relative Strength Index) or MACD
3. **Change the prediction horizon** — instead of 5 days, try predicting 10 or 20 days ahead
4. **Try a different model** — replace `GradientBoostingClassifier` with `RandomForestClassifier`
5. **Add a simple backtester** — start with $10,000, buy on signal=1, sell on signal=-1, track portfolio value

---

## 📚 Summary: Concepts Learned

| Concept | Plain English Explanation |
|---------|---------------------------|
| **Simple Moving Average (SMA)** | Average of the last N closing prices, equally weighted |
| **Exponential Moving Average (EMA)** | Weighted average where recent prices count more |
| **SMA Crossover** | Trading signal generated when short SMA crosses over long SMA |
| **Momentum** | How much price changed over a specific number of days |
| **Volatility** | Standard deviation of daily returns — measures how jumpy the stock is |
| **Gradient Boosting** | Ensemble of weak learners (trees) that correct each other's mistakes |
| **Feature Engineering** | Transforming raw data into informative inputs for an ML model |
| **Time-series split** | Splitting data without shuffling to avoid 'looking at the future' |
| **StandardScaler** | Normalises features to zero mean and unit variance |
| **GBM (Geometric Brownian Motion)** | Mathematical model for simulating realistic stock prices |